## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import mujoco

from IPython.display import Video, display

from src.model.arm import Arm
from src.sim.simulation_io import (load_simulations, plot_activations, plot_angles, plot_hand_error,
                                   plot_hand_path, plot_hand_speed, plot_torques, simulation_path, summary_table)

print(f"MuJoCo {mujoco.__version__}")

## Parameters

In [ ]:
PATH_MUJOCO_MODEL = "assets/myo_sim/arm/myoarm.xml"
RESOLUTION_MS = 1.0

TRAJ_NAME = "minjerk_500ms"   # Trajectory.name: "minjerk_500ms", "cartesian_500ms", "step"
THETA_START = [-20.0, 20.0]
THETA_END = [70.0, 90.0]
W_ref = 0.5
P_VALUE = 3
K_P = [4.0, 2.0]
K_I = [2.0, 1.0]
K_D = [0.1, 0.05]
DELAY_MS = 0.0
FEEDFORWARD = True

LABELS = None # ["solver"]

SAVE_PLOTS = False
PLOT_T_MIN = None
PLOT_T_MAX = None
STEP_MS = 10
PLAYER_VIEW = ["top", "front", "side"]
VIDEO_VIEW = ["top", "front"]
VIDEO_QUALITY = "full" # "fast", "normal", "full"

experiment = dict(traj_name=TRAJ_NAME, theta_start_deg=THETA_START, theta_end_deg=THETA_END, W=W_ref,
                  p=P_VALUE, K_P=K_P, K_I=K_I, K_D=K_D, delay_ms=DELAY_MS, feedforward=FEEDFORWARD)
SIM_DIR = simulation_path("simulations/arm", **experiment)
PLOT_DIR = simulation_path("plots/arm", **experiment) if SAVE_PLOTS else None
VIDEO_DIR = simulation_path("videos/arm", **experiment)
print(SIM_DIR)

## Load

In [ ]:
arm = Arm.planar(PATH_MUJOCO_MODEL)
sims = load_simulations(SIM_DIR, arm, labels=LABELS)
sim = next(iter(sims.values()))
summary_table(sims).T

## Trajectory

In [ ]:
sim.traj.plot(show_phases=True, save_path=f"{PLOT_DIR}/trajectory.png" if SAVE_PLOTS else None)
sim.traj.plot_hand(save_path=f"{PLOT_DIR}/trajectory_hand.png" if SAVE_PLOTS else None)
sim.traj.plot_views(PLAYER_VIEW, save_path=f"{PLOT_DIR}/trajectory_views.png" if SAVE_PLOTS else None)

## Plots

In [ ]:
window = dict(t_min=PLOT_T_MIN, t_max=PLOT_T_MAX)
plot_angles(sims, save_dir=PLOT_DIR, **window)
plot_torques(sims, save_dir=PLOT_DIR, **window)
# plot_activations(sims, save_dir=PLOT_DIR, **window)
# plot_activations(sims, state=True, save_dir=PLOT_DIR, **window)

## Hand

In [ ]:
plot_hand_path(sims, save_dir=PLOT_DIR)
plot_hand_speed(sims, save_dir=PLOT_DIR)
# plot_hand_error(sims, save_dir=PLOT_DIR)

## Interactive player

In [ ]:
model = arm.simulation_model(timestep_ms=RESOLUTION_MS)
sim.player(model, step_ms=STEP_MS, view=PLAYER_VIEW)

## Save Player

In [ ]:
sim.save_player(model, f"{VIDEO_DIR}/{sim.label}_player.html", step_ms=STEP_MS, view=PLAYER_VIEW)

## Video

In [ ]:
# video_path = f"{VIDEO_DIR}/{sim.label}_video.mp4"
# sim.save_video(model, video_path, fps=30, slowdown=4, view=VIDEO_VIEW, quality=VIDEO_QUALITY)
# display(Video(video_path, embed=True, width=960))